# *ETL*

In [17]:
# Importo las librerias que usaré
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from unidecode import unidecode

In [18]:
# Cargo ruta de la data
integrantes_excel = 'C:/Users/delfi/MisProyectosPython/p2/bd_integrantes_Esp.xlsx'
# Cargo el archivo de Excel en un DataFrame de Pandas
df_integrantes = pd.read_excel(integrantes_excel)

In [4]:
# Visualizo el df_integrantes para saber que debo transformar y/o limpiar
df_integrantes.head(10)

,Nombre de los miembros,ID,Salario-Cargo,Fecha Nacimiento,Edad,Sexo,Fecha de contratación,Fecha salida,Status do Integrante,Departamento,Reclutamiento de fuentes,Registro de desempeño,Índice de satisfacción
0,Isis,A1304055947,5060-Técnico de Producción I,1982-05-21,38,F,2015-09-01,NaN,Ativo,dpt1,Página Web compañía,Arriba,5.0
1,Emanuel,A1109029366,3520-Técnico de Producción I,1975-12-21,45,M,2015-02-03,NaN,Ativo,dpt1,Recomendación empleados,Dentro de lo esperado,3.0
2,Vinícius,A1501072124,4400-Técnico de Producción I,1972-04-17,49,M,2017-07-06,NaN,Ativo,dpt1,Sitio web de empleo,Abajo de lo esperado,2.0
3,João Guilherme,A1302053339,3960-Técnico de Producción I,1994-10-17,26,M,2018-11-04,NaN,Ativo,dpt1,Sitio web de empleo,Abajo de lo esperado,4.0
4,Luna,A1204032927,3520-Técnico de Producción I,1985-08-04,35,F,2017-09-28,NaN,Ativo,dpt1,Feria de contratación,Arriba,4.0
5,Rebeca,A1404066622,4400-Técnico de Producción I,1979-02-04,42,F,2017-12-04,NaN,Ativo,dpt1,Sitio web de empleo,Arriba,4.0
6,Maria,A1205033102,3300-Técnico de Producción I,1990-08-23,30,F,2016-08-18,NaN,Ativo,dpt1,Sitio web de empleo,Dentro de lo esperado,4.0
7,Catarina,A1308060366,3520-Técnico de Producción I,1994-01-08,27,F,2017-07-06,NaN,Ativo,dpt1,Feria de contratación,Arriba,3.1
8,Luiz Miguel,A1407069280,5445-Técnico de Producción I,1985-08-25,35,M,2019-06-07,NaN,Ativo,dpt1,Recomendación empleados,Arriba,4.0
9,Clara,A1209048696,4840-Técnico de Producción I,1983-02-11,38,F,2016-06-30,NaN,Ativo,dpt1,Sitio web de empleo,Abajo de lo esperado,3.0


In [5]:
# Voy a normalizar el type de los datos, para sea el adecuado en cada uno
df_integrantes.info()

<class 'pandas.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Nombre de los miembros    207 non-null    str           
 1   ID                        207 non-null    str           
 2   Salario-Cargo             207 non-null    str           
 3   Fecha Nacimiento          207 non-null    datetime64[us]
 4   Edad                      207 non-null    int64         
 5   Sexo                      207 non-null    str           
 6   Fecha de contratación     207 non-null    datetime64[us]
 7   Fecha salida              0 non-null      float64       
 8   Status do Integrante      207 non-null    str           
 9   Departamento              207 non-null    str           
 10  Reclutamiento de fuentes  207 non-null    str           
 11  Registro de desempeño     207 non-null    str           
 12  Índice de satisfacción    207 non

## Análisis inicial y transformaciones ETL

Durante la etapa de *data cleaning* del proceso ETL, se aplican las siguientes transformaciones sobre la base de datos:

- Eliminación de tildes en los campos de tipo `string`, con el objetivo de evitar inconsistencias y errores durante la carga de datos en Power BI.
- Normalización del campo `Salario-Cargo`, el cual contiene más de una variable en un mismo atributo; se procede a su separación en columnas independientes.
- Eliminación del campo `Fecha salida`, ya que no contiene registros y no aporta valor analítico para esta etapa del proceso.
- Corrección del nombre del campo `Status do Integrante` a `Estatus de Integrante`, y reemplazo del valor `Ativo` por `Activo`.
- Estandarización de los valores del campo `Registro de desempeño`, reemplazando el valor ambiguo `Arriba` por `Arriba de lo esperado`.
- Conversión y normalización de los tipos de datos de cada columna, asegurando su adecuación para el modelo de datos y su posterior carga.


In [20]:
# Eliminar asteriscos en todo el DataFrame
columnas_str = df_integrantes.select_dtypes(include='object').columns

for columna in columnas_str:
    df_integrantes[columna] = df_integrantes[columna].map(lambda x: unidecode(x) 
                                                          if isinstance(x, str) 
                                                          else x)

C:\Users\delfi\AppData\Local\Temp\ipykernel_3996\3267479546.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_str = df_integrantes.select_dtypes(include='object').columns


In [21]:
# Separar `Salario-Cargo`
df_integrantes[['Salario', 'Cargo']] = df_integrantes['Salario-Cargo'].str.split('-', expand=True)

df_integrantes['Salario'] = df_integrantes['Salario'].str.strip().astype(float)
df_integrantes['Cargo'] = df_integrantes['Cargo'].str.strip()

df_integrantes.drop(columns='Salario-Cargo', inplace=True)

In [22]:
# Eliminar columna sin valor analítico
df_integrantes.drop(columns='Fecha salida', inplace=True)

In [23]:
# Corregir nombre y valores
df_integrantes.rename(columns={'Status do Integrante': 'Estatus de Integrante'}, inplace=True)

df_integrantes.replace('Ativo', 'Activo', inplace=True)


,Nombre de los miembros,ID,Fecha Nacimiento,Edad,Sexo,Fecha de contratación,Estatus de Integrante,Departamento,Reclutamiento de fuentes,Registro de desempeño,Índice de satisfacción,Salario,Cargo
0,Isis,A1304055947,1982-05-21,38,F,2015-09-01,Activo,dpt1,Pagina Web compania,Arriba,5.0,5060.0,Tecnico de Produccion I
1,Emanuel,A1109029366,1975-12-21,45,M,2015-02-03,Activo,dpt1,Recomendacion empleados,Dentro de lo esperado,3.0,3520.0,Tecnico de Produccion I
2,Vinicius,A1501072124,1972-04-17,49,M,2017-07-06,Activo,dpt1,Sitio web de empleo,Abajo de lo esperado,2.0,4400.0,Tecnico de Produccion I
3,Joao Guilherme,A1302053339,1994-10-17,26,M,2018-11-04,Activo,dpt1,Sitio web de empleo,Abajo de lo esperado,4.0,3960.0,Tecnico de Produccion I
4,Luna,A1204032927,1985-08-04,35,F,2017-09-28,Activo,dpt1,Feria de contratacion,Arriba,4.0,3520.0,Tecnico de Produccion I
...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,Leticia Barbosa,A1411071312,1957-01-16,54,F,2017-10-10,Activo,dpt4,Recomendacion empleados,Dentro de lo esperado,5.0,11902.0,Ingeniero de software
203,Bernardo S.,A1108028108,1991-04-22,30,M,2017-10-10,Activo,dpt4,Recomendacion empleados,Dentro de lo esperado,5.0,12364.0,Ingeniero de software
204,Marcos de F.,A904013591,1988-08-31,32,M,2019-06-30,Activo,dpt4,Feria de contratacion,Dentro de lo esperado,4.0,11836.0,Ingeniero de software
205,Mario Andradde,A1308060959,1970-09-08,50,M,2017-10-10,Activo,dpt4,Recomendacion empleados,Dentro de lo esperado,3.0,11660.0,Ingeniero de software


In [24]:
# Corregir desempeño
df_integrantes.replace('Arriba', 'Arriba de lo esperado', inplace=True)


,Nombre de los miembros,ID,Fecha Nacimiento,Edad,Sexo,Fecha de contratación,Estatus de Integrante,Departamento,Reclutamiento de fuentes,Registro de desempeño,Índice de satisfacción,Salario,Cargo
0,Isis,A1304055947,1982-05-21,38,F,2015-09-01,Activo,dpt1,Pagina Web compania,Arriba de lo esperado,5.0,5060.0,Tecnico de Produccion I
1,Emanuel,A1109029366,1975-12-21,45,M,2015-02-03,Activo,dpt1,Recomendacion empleados,Dentro de lo esperado,3.0,3520.0,Tecnico de Produccion I
2,Vinicius,A1501072124,1972-04-17,49,M,2017-07-06,Activo,dpt1,Sitio web de empleo,Abajo de lo esperado,2.0,4400.0,Tecnico de Produccion I
3,Joao Guilherme,A1302053339,1994-10-17,26,M,2018-11-04,Activo,dpt1,Sitio web de empleo,Abajo de lo esperado,4.0,3960.0,Tecnico de Produccion I
4,Luna,A1204032927,1985-08-04,35,F,2017-09-28,Activo,dpt1,Feria de contratacion,Arriba de lo esperado,4.0,3520.0,Tecnico de Produccion I
...,...,...,...,...,...,...,...,...,...,...,...,...,...
202,Leticia Barbosa,A1411071312,1957-01-16,54,F,2017-10-10,Activo,dpt4,Recomendacion empleados,Dentro de lo esperado,5.0,11902.0,Ingeniero de software
203,Bernardo S.,A1108028108,1991-04-22,30,M,2017-10-10,Activo,dpt4,Recomendacion empleados,Dentro de lo esperado,5.0,12364.0,Ingeniero de software
204,Marcos de F.,A904013591,1988-08-31,32,M,2019-06-30,Activo,dpt4,Feria de contratacion,Dentro de lo esperado,4.0,11836.0,Ingeniero de software
205,Mario Andradde,A1308060959,1970-09-08,50,M,2017-10-10,Activo,dpt4,Recomendacion empleados,Dentro de lo esperado,3.0,11660.0,Ingeniero de software


In [26]:
df_integrantes['Salario'] = pd.to_numeric(df_integrantes['Salario'], errors='coerce')